In [1]:
%pip install -q "transformers==4.57.1" "accelerate==1.11.0" "peft==0.17.1"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import gc
import hashlib
import re
import unicodedata
from pathlib import Path
from decimal import Decimal
import json
import torch

from peft import PeftModel
from safetensors.torch import load_file
from transformers import GenerationConfig, Qwen3VLForConditionalGeneration, AutoProcessor

In [4]:
MLLM = Path("/content/drive/MyDrive/MLLM/mordern_llm")
D_ROOT = Path("/content/drive/MyDrive/MLLM/section_d")
D_DATA = D_ROOT / "data"
D_DATA.mkdir(parents=True, exist_ok=True)

C_DATA = MLLM / "data/sroie"
manifest = json.loads((C_DATA / "manifest.json").read_text(encoding="utf-8"))
task = json.loads((C_DATA / "task_config.json").read_text(encoding="utf-8"))

TRAIN = manifest["train"]
FIELDS = tuple(task["fields"])
DOC_PROMPT = task["prompt"]
IMAGE_KWARGS = task["image_kwargs"]

processor = AutoProcessor.from_pretrained(MLLM / "outputs/qlora_run1/best_adapter", use_fast=False)
processor.tokenizer.padding_side = "right"

print("Section C folder:", MLLM)
print("Train / validation:", len(TRAIN), len(manifest["val"]))
print("Fields:", FIELDS)

Section C folder: /content/drive/MyDrive/MLLM/mordern_llm
Train / validation: 551 61
Fields: ('company', 'date', 'total')


In [ ]:
def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

PILOT = D_ROOT / "outputs/sft_pilot_v1"
OUT = D_ROOT / "outputs/day4_heldout"
OUT.mkdir(parents=True, exist_ok=True)

recipe_path = PILOT / "recipe.json"
lock_path = D_ROOT / "data/heldout_manifest.json"

recipe = read_json(recipe_path)
heldout = read_json(lock_path)
TEST = heldout["test"]
TEST_IDS = [r["id"] for r in TEST]

used = manifest["train"] + manifest["val"] + manifest["test"]
used_ids = {r["id"] for r in used}
used_hashes = {r["sha256"] for r in used}
test_hashes = {r["sha256"] for r in TEST}

task = recipe["task"]
FIELDS = tuple(task["fields"])
DOC_PROMPT = task["prompt"]
EVAL_BUDGET = recipe["image_kwargs"]

DTYPE = {
    "torch.float16": torch.float16,
    "torch.bfloat16": torch.bfloat16
}[recipe["dtype"]]

DEVICE = torch.device("cuda:0")
ADAPTER = PILOT / "llm_lora/best_adapter"
MERGER = PILOT / "projector_only/best_merger.safetensors"
CHECKPOINT_STEP = {
    name: read_json(PILOT / name / "best_validation.json")["step"]
    for name in ("llm_lora", "projector_only")
}

print("Held-out receipts:", len(TEST))
print("Selected steps:", CHECKPOINT_STEP)
print("Dtype:", DTYPE)

Held-out receipts: 30
Selected steps: {'llm_lora': 8, 'projector_only': 8}
Dtype: torch.float16


In [6]:
def normal(field, value):
    if not isinstance(value, str) or not value.strip():
        return None

    text = " ".join(unicodedata.normalize("NFKC", value).casefold().split())
    if field != "total":
        return text
    
    text = re.sub(r"^(?:rm|\$)\s*", "", text)
    if not re.fullmatch(
        r"[+-]?(?:\d+|\d{1,3}(?:,\d{3})+)(?:\.\d+)?", text
    ):
        return None

    amount = Decimal(text.replace(",", ""))
    return format(amount.normalize(), "f")


def score_doc(answer, reference):
    raw_valid = True
    used_cleanup = False

    try:
        prediction = json.loads(answer)
    except json.JSONDecodeError:
        raw_valid = False
        prediction = None

        fence = re.fullmatch(r"```(?:json)?\s*(.*?)\s*```", answer.strip(), flags=re.IGNORECASE | re.DOTALL)

        if fence:
            used_cleanup = True
            try:
                prediction = json.loads(fence.group(1))
            except json.JSONDecodeError:
                pass

    schema_valid = (
        isinstance(prediction, dict)
        and set(prediction) == set(FIELDS)
        and all(value is None or isinstance(value, str) for value in prediction.values())
    )

    flags = {}

    for field in FIELDS:
        correct = False

        if schema_valid:
            predicted = prediction[field]
            expected = reference[field]

            if predicted is None or expected is None:
                correct = predicted is None and expected is None
            else:
                normalized = normal(field, predicted)
                correct = (
                    normalized is not None
                    and normalized == normal(field, expected)
                )

        flags[f"{field}_correct"] = correct

    return {
        "prediction": prediction,
        "raw_json_valid": raw_valid,
        "schema_valid": schema_valid,
        "used_fence_cleanup": used_cleanup,
        **flags,
        "all_correct": all(flags.values())
    }